# Replicando análisis de FoodProK en Stata


## Notas de MARKDOWN 

# Título grande
## Título mediano
### Título pequeño

Texto normal en párrafo

**texto en negritas**

*texto en cursiva*

- punto de lista
- otro punto

In [1]:
# Importar librerías cada vez que quiera usar Python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pedirle print para verificar que todo se instaló correctamente
print("Todo instalado correctamente ✓")
print(f"Pandas versión: {pd.__version__}")

Todo instalado correctamente ✓
Pandas versión: 3.0.1


# REVISAR ARCHIVOS E IMPORTAR BASES DE DATOS

In [2]:
# Verificar Nombre de archivos antes de importar las bases
import os


# Mi notebook está guardado en la carpeta notebooks
## Iniciar el path con ../ para indicarle a Python que "suba" un nivel en los folders
archivos = os.listdir("../data/raw/")
print(archivos)

['IFPS2020_allcountry_analytic_20230718.dta', 'IFPS2021_allcountry_analytic_20230717.dta', 'IFPS2022_allcountry_analytic_20230830.dta']


In [3]:
# Cargar bases de datos como data frames

# Cargar 2020
df_2020 = pd.read_stata("../data/raw/IFPS2020_allcountry_analytic_20230718.dta", convert_categoricals=False)

    ## Stata permite etiquetas de valor duplicadas, python NO.
    ## convert_categoricals=False evita este error

# Cargar 2021
df_2021 = pd.read_stata(
    "../data/raw/IFPS2021_allcountry_analytic_20230717.dta",
    convert_categoricals=False
)

# Cargar 2022
df_2022 = pd.read_stata(
    "../data/raw/IFPS2022_allcountry_analytic_20230830.dta",
    convert_categoricals=False
)

In [4]:
# Ver y print cantidad de filas y columnas antes de unir
print(f"2020: {df_2020.shape}")
print(f"2021: {df_2021.shape}")
print(f"2022: {df_2022.shape}")

2020: (21753, 1833)
2021: (26285, 1828)
2022: (26273, 1948)


Nota: Cada base tiene diferente número de observaciones (porque no es longitudinal) y diferente número de variables (se agregan o eliminan preguntas cada año)

In [5]:
# Ver solo los nombres de todas las columnas en 2020 (como referencia)
df_2020.columns.tolist()

['ID',
 'wght',
 'W3W4_match_DV',
 'W3_ID_match_DV',
 'W2W3W4_match_DV',
 'W2_ID_match_DV',
 'W1W2W3W4_match_DV',
 'W1_ID_match_DV',
 'country',
 'sample_USA',
 'language',
 'Vdatesub',
 'TimeStarted',
 'DateSubmitted',
 'monthsub_DV',
 'status_DV',
 'respstatus_DV',
 'expletive_DV',
 'mobilebrowser',
 'age',
 'age_DV',
 'sex',
 'consent',
 'gender',
 'gender_otext',
 'gender_DV',
 'student',
 'occup',
 'occup_otext',
 'occup_DV',
 'occup_covid_DV',
 'child_any',
 'child_home',
 'child_home_DV',
 'child_home_U18_DV',
 'child_hhld_DV',
 'child_DQ_DV',
 'child1_age',
 'child1_age_DKR',
 'child2_age',
 'child2_age_DKR',
 'child3_age',
 'child3_age_DKR',
 'child4_age',
 'child4_age_DKR',
 'child5_age',
 'child5_age_DKR',
 'child6_age',
 'child6_age_DKR',
 'child7_age',
 'child7_age_DKR',
 'child8_age',
 'child8_age_DKR',
 'child9_age',
 'child9_age_DKR',
 'child10_age',
 'child10_age_DKR',
 'child1_age_DV',
 'child2_age_DV',
 'child3_age_DV',
 'child4_age_DV',
 'child5_age_DV',
 'child6_ag

In [6]:
# Explorar AÑO
print('year' in df_2020.columns)
print('year' in df_2021.columns)
print('year' in df_2022.columns)

False
False
False


Nota: Al explorar las bases, confirmé que a variable "year" arrojó FALSE en los 3 años.

Esto significa que la variable AÑO no existe en las bases.

La creo manualmente antes de unir para poder identificar cada observación por año después.

In [7]:
# Crear variable year en cada base antes de unir
df_2020['year'] = 2020
df_2021['year'] = 2021
df_2022['year'] = 2022

C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3571457639.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_2020['year'] = 2020
C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3571457639.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_2021['year'] = 2021
C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3571457639.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conca

In [8]:
# Unir bases de datos (equivalente al append de Stata)

df = pd.concat([df_2020, df_2021, df_2022], ignore_index=True)
    # ignore_index=True reinicia la numeración de filas desde 0

# Verificar que quedó bien
print(df.shape)
print(df['year'].value_counts())

(74311, 2210)
year
2021    26285
2022    26273
2020    21753
Name: count, dtype: int64


La base unificada tiene 74,311 participantes y 2,210 columnas

2021: **26,285** → participantes de 2021

2022: **26,273** → participantes de 2022

2020: **21,753** → participantes de 2020

Suma: **21,753** + 26,285 + 26,273 = **74,311**

# Explorar variables de interés
## País

In [9]:
# Explorar variable Country
print(df['country'].value_counts())
print(df['country'].value_counts(normalize=True).round(3) * 100)

country
4.0    19390
5.0    16355
1.0    13318
3.0    12648
2.0    12600
Name: count, dtype: int64
country
4.0    26.1
5.0    22.0
1.0    17.9
3.0    17.0
2.0    17.0
Name: proportion, dtype: float64


Nota:

1 Canada 13318 obs

2 Australia 12600 obs

3 UK 12648 obs

4 USA 19390 obs

5 Mexico 16355 obs

Total: 74311 obs

In [10]:
# Convertir códigos numéricos a nombres de países
df['country'] = df['country'].astype("Int64")

country_labels = {1: 'Canada', 2: 'Australia', 3: 'UK', 4: 'USA', 5: 'Mexico'}
df['country'] = df['country'].map(country_labels)
    # .map() Es el equivalente del recode + label define de Stata —> toma cada valor numérico y lo reemplaza con su etiqueta.

# Definir orden alfabético con Australia como referencia
df['country'] = pd.Categorical(
    df['country'],
    categories=['Australia', 'Canada', 'Mexico', 'UK', 'USA'],
    ordered=False
)

# Verificar — debe coincidir con do file
print(df['country'].cat.categories)   # Muestra el orden asignado
print(df['country'].value_counts().sort_index())  # Imprimir valores con el orden del índice - Los valores deben coincidir con el do file

Index(['Australia', 'Canada', 'Mexico', 'UK', 'USA'], dtype='str')
country
Australia    12600
Canada       13318
Mexico       16355
UK           12648
USA          19390
Name: count, dtype: int64


## Edad

In [11]:
# Explorar variable age
print(df['age'].describe())  # .describe da media, desviación estándar, mínimo, máximo y percentile
print(f"\nMissings: {df['age'].isnull().sum()}") ## isnull para identificar los missing

count    74311.000000
mean        45.361010
std         16.850802
min         18.000000
25%         31.000000
50%         45.000000
75%         59.000000
max         99.000000
Name: age, dtype: float64

Missings: 0


Note: No hay missings en Edad

## Sexo

In [12]:
# Explorar variable sex
print(df['sex'].value_counts())
print(f"\nMissings: {df['sex'].isnull().sum()}")

sex
2.0    38043
1.0    36268
Name: count, dtype: int64

Missings: 0


Nota: No hay missings en sexo

In [13]:
# Mapear sexo a texto
sex_labels = {1: 'Male', 2: 'Female'}
df['sex'] = df['sex'].map(sex_labels)

# Verificar
print(df['sex'].value_counts())

sex
Female    38043
Male      36268
Name: count, dtype: int64


## Educación

In [14]:
# Explorar variable education
print(df['educ_DV'].value_counts())
print(f"\nMissings: {df['educ_DV'].isnull().sum()}")

educ_DV
 3.0     28592
 1.0     26411
 2.0     18986
-99.0      322
Name: count, dtype: int64

Missings: 0


In [15]:
# Guardar tamaño antes
n_antes = len(df)

# Eliminar "Not stated" (-99)
df = df[df['educ_DV'] != -99]

# Verificar cuántos se eliminaron
# Mostrar cuántos se eliminaron automáticamente
print(f"Eliminados: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Mapear a texto con orden lógico
educ_labels = {1: 'Low', 2: 'Medium', 3: 'High'}
df['educ_DV'] = df['educ_DV'].map(educ_labels)

# Definir orden lógico (no alfabético — aquí el orden importa)
df['educ_DV'] = pd.Categorical(
    df['educ_DV'],
    categories=['Low', 'Medium', 'High'],
    ordered=True
)

print(df['educ_DV'].value_counts().sort_index())

Eliminados: 322
Observaciones restantes: 73989
educ_DV
Low       26411
Medium    18986
High      28592
Name: count, dtype: int64


## Etnicidad

In [16]:
# Explorar etnicidad
print(df['eth_DV'].value_counts())
print(f"\nMissings: {df['eth_DV'].isnull().sum()}")

eth_DV
 1.0     53275
 2.0     19989
-99.0      725
Name: count, dtype: int64

Missings: 0


In [17]:
# Guardar tamaño antes
n_antes = len(df)

# Eliminar "Not stated" (-99)
df = df[df['eth_DV'] != -99]

# Verificar cuántos se eliminaron
# Mostrar cuántos se eliminaron automáticamente
print(f"Eliminados: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Mapear a texto con orden lógico
eth_labels = {1: 'Majority', 2: 'Minority'}
df['eth_DV'] = df['eth_DV'].map(eth_labels)

# Definir orden lógico 
df['eth_DV'] = pd.Categorical(
    df['eth_DV'],
    categories=['Majority', 'Minority'],
    ordered=False
)

print(df['eth_DV'].value_counts().sort_index())

Eliminados: 725
Observaciones restantes: 73264
eth_DV
Majority    53275
Minority    19989
Name: count, dtype: int64


## Income Adequacy

In [18]:
# Explorar Income adequacy
print(df['income_adeq'].value_counts())
print(f"\nMissings: {df['income_adeq'].isnull().sum()}")

income_adeq
 3.0     26656
 2.0     15630
 4.0     15410
 5.0      8597
 1.0      6293
-77.0      415
-88.0      263
Name: count, dtype: int64

Missings: 0


In [19]:
# Guardar tamaño antes
n_antes = len(df)

# Eliminar DK y RTA (-77, -88)
df = df[df['income_adeq'] != -88]
df = df[df['income_adeq'] != -77]

# Verificar cuántos se eliminaron
# Mostrar cuántos se eliminaron automáticamente
print(f"Eliminados: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Recodificar 5 categorías a 3
income_labels = {
    1: 'Difficult',
    2: 'Difficult',
    3: 'Neither',
    4: 'Easy',
    5: 'Easy'
}
df['income_adeq'] = df['income_adeq'].map(income_labels)

# Definir orden lógico
df['income_adeq'] = pd.Categorical(
    df['income_adeq'],
    categories=['Difficult', 'Neither', 'Easy'],
    ordered=True
)

# Verificar
print(df['income_adeq'].value_counts().sort_index())

Eliminados: 678
Observaciones restantes: 72586
income_adeq
Difficult    21923
Neither      26656
Easy         24007
Name: count, dtype: int64


# Limpiar y Construir FoodProK

In [20]:
# Explorar las variables hlth que usaremos
    # Crear una lista con las variables
hlth_vars = ['hlth1_DV', 'hlth3_DV', 'hlth4_DV', 'hlth6_DV', 
             'hlth7_DV', 'hlth9_DV', 'hlth10_DV', 'hlth11_DV']

for var in hlth_vars:
    print(f"\n{var}:")
    print(df[var].value_counts().sort_index())


hlth1_DV:
hlth1_DV
-88.0       32
-77.0      427
 0.0       112
 1.0        74
 2.0       135
 3.0       232
 4.0       391
 5.0      1441
 6.0      1574
 7.0      4068
 8.0      8824
 9.0     12594
 10.0    42514
Name: count, dtype: int64

hlth3_DV:
hlth3_DV
-88.0       46
-77.0     1458
 0.0      5503
 1.0      3217
 2.0      5747
 3.0      7551
 4.0      7663
 5.0     10862
 6.0      8119
 7.0      7981
 8.0      6649
 9.0      3678
 10.0     3933
Name: count, dtype: int64

hlth4_DV:
hlth4_DV
-88.0       47
-77.0     1575
 0.0       541
 1.0       314
 2.0       683
 3.0      1312
 4.0      2154
 5.0      5904
 6.0      6296
 7.0     10853
 8.0     15131
 9.0     12549
 10.0    15038
Name: count, dtype: int64

hlth6_DV:
hlth6_DV
-88.0       61
-77.0     2259
 0.0      6592
 1.0      3815
 2.0      6579
 3.0      8591
 4.0      8716
 5.0     11716
 6.0      7765
 7.0      6730
 8.0      5024
 9.0      2214
 10.0     2343
Name: count, dtype: int64

hlth7_DV:
hlth7_DV
-88.0       51
-

In [21]:
print(df.shape)
print(df['country'].dtype)

(72586, 2210)
category


In [22]:
# Codebook de variables hlth — para referencia durante el análisis
    # Crear diccionario
hlth_codebook = {
    'hlth1_DV':  'Apple (unprocessed)',
    'hlth3_DV':  'Apple fruit drink (processed)',
    'hlth4_DV':  'Oats (unprocessed)',
    'hlth6_DV':  'Oat breakfast cereal (processed)',
    'hlth7_DV':  'Milk (unprocessed)',
    'hlth9_DV':  'Processed cheese slice (processed)',
    'hlth10_DV': 'Chicken breast (unprocessed)',
    'hlth11_DV': 'Chicken nuggets (processed)'
}

# Para consultar cualquier variable:
print(hlth_codebook['hlth1_DV'])

Apple (unprocessed)


## Validar FoodProK (respondió IDK o RTA a 5 o más)

In [23]:
# Contar cuántas veces respondió IDK (-77) en las 8 variables

df['idk_count'] = (df[hlth_vars] == -77).sum(axis=1)
    # Revisa cada celda de la lista de variables y marca True si esa celda es -77
    # .sum(axis=1) suma los True de cada fila
    # El valor será el número de IDK de cada persona


# Ver la distribución (Cuántas personas respondieron IDK a 1,2...8 hlth_vars)
print(df['idk_count'].value_counts().sort_index())

idk_count
0    68340
1     1735
2      698
3      400
4      325
5      249
6      213
7      366
8      260
Name: count, dtype: int64


C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\1472112288.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['idk_count'] = (df[hlth_vars] == -77).sum(axis=1)


In [24]:
# Eliminar personas con 5 o más respuestas IDK
# Guardar tamaño antes
n_antes = len(df)

# Conservar IDK <5
df = df[df['idk_count'] < 5]

# Verificar cuántos se eliminaron
print(f"Eliminados por IDK>5: {n_antes - len(df)}")
# Nuevo sample size
print(f"Observaciones restantes: {len(df)}")

# 1088 eliminados

Eliminados por IDK>5: 1088
Observaciones restantes: 71498


## Eliminar RTA y missings

In [25]:
# Eliminar personas con -88 (RTA) en cualquier variable hlth

#Guardar tamaño antes
n_antes = len(df)

# Conservar
df = df[~(df[hlth_vars] == -88).any(axis=1)]
    # .any(axis=1) devuelve True si al menos una columna de esa fila tiene -88
    # ~ es el "NO" en Python (invierte el True/False) --> Quédate con las filas donde NO hay ningún -88 (=! funciona solo para 1 columna)

print(f"Eliminados por RTA: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Eliminados 184

Eliminados por RTA: 184
Observaciones restantes: 71314


In [26]:
# Ver si hay missings reales (NaN) en las variables hlth
print(df[hlth_vars].isnull().sum())

hlth1_DV     165
hlth3_DV     178
hlth4_DV     187
hlth6_DV     180
hlth7_DV     210
hlth9_DV     192
hlth10_DV    206
hlth11_DV    186
dtype: int64


In [27]:
# Eliminar missings restantes
#Guardar tamaño antes
n_antes = len(df)

# Eliminar missings
df = df.dropna(subset=hlth_vars)
    # dropna Elimina filas que tengan NaN en cualquiera de las variables de la lista

print(f"Eliminados por Missings: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Eliminados 1389

Eliminados por Missings: 1389
Observaciones restantes: 69925


## Construir categorías de FoodProK

In [28]:
# Construir los 4 pares
df['fruit']   = (df['hlth1_DV'] > df['hlth3_DV']).astype(int)
df['cereal']  = (df['hlth4_DV'] > df['hlth6_DV']).astype(int)
df['dairy']   = (df['hlth7_DV'] > df['hlth9_DV']).astype(int)
df['chicken'] = (df['hlth10_DV'] > df['hlth11_DV']).astype(int)
    # Como hay personas que respondieron 

# Verificar distribución de cada par
for var in ['fruit', 'cereal', 'dairy', 'chicken']:
    print(f"\n{var}:")
    print(df[var].value_counts().sort_index())


fruit:
fruit
0     8856
1    61069
Name: count, dtype: int64

cereal:
cereal
0    14755
1    55170
Name: count, dtype: int64

dairy:
dairy
0    20204
1    49721
Name: count, dtype: int64

chicken:
chicken
0    11444
1    58481
Name: count, dtype: int64


C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3529284500.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['fruit']   = (df['hlth1_DV'] > df['hlth3_DV']).astype(int)
C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3529284500.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['cereal']  = (df['hlth4_DV'] > df['hlth6_DV']).astype(int)
C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3529284500.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

In [29]:
# Si respondió IDK (-77) en cualquier variable del par → 0
    #.loc es la forma de reemplazar valores en Pandas (como replace en Stata)
df.loc[df['hlth1_DV'] == -77, 'fruit'] = 0
df.loc[df['hlth3_DV'] == -77, 'fruit'] = 0

df.loc[df['hlth4_DV'] == -77, 'cereal'] = 0
df.loc[df['hlth6_DV'] == -77, 'cereal'] = 0

df.loc[df['hlth7_DV'] == -77, 'dairy'] = 0
df.loc[df['hlth9_DV'] == -77, 'dairy'] = 0

df.loc[df['hlth10_DV'] == -77, 'chicken'] = 0
df.loc[df['hlth11_DV'] == -77, 'chicken'] = 0

# Verificar
for var in ['fruit', 'cereal', 'dairy', 'chicken']:
    print(f"\n{var}:")
    print(df[var].value_counts().sort_index())


fruit:
fruit
0     9337
1    60588
Name: count, dtype: int64

cereal:
cereal
0    15722
1    54203
Name: count, dtype: int64

dairy:
dairy
0    21113
1    48812
Name: count, dtype: int64

chicken:
chicken
0    12012
1    57913
Name: count, dtype: int64


## Construir FoodProK total

In [30]:
# Construir foodprok --> suma de los 4 pares (0 a 4)
df['foodprok'] = df['fruit'] + df['cereal'] + df['dairy'] + df['chicken']

# Verificar distribución y media
print(df['foodprok'].value_counts().sort_index())
print(f"\nMedia: {df['foodprok'].mean().round(2)}")

foodprok
0     2237
1     4236
2     8348
3    19832
4    35272
Name: count, dtype: int64

Media: 3.17


C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\3957945486.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['foodprok'] = df['fruit'] + df['cereal'] + df['dairy'] + df['chicken']


In [31]:
# Explorar AÑO
print('year' in df_2020.columns)
print('year' in df_2021.columns)
print('year' in df_2022.columns)
print(df['country'].cat.categories)

True
True
True
Index(['Australia', 'Canada', 'Mexico', 'UK', 'USA'], dtype='str')


# Reescalar Weights

In [32]:
print(df['wght'].describe())

count    69925.000000
mean         0.996690
std          0.748694
min          0.032878
25%          0.531507
50%          0.842440
75%          1.232380
max          5.128250
Name: wght, dtype: float64


In [33]:
# Reescalar pesos por país y año separadamente

# Formula: new weight = old weight x analytic sample size / sum of the old weights
df['wght_rescaled'] = df.groupby(['country', 'year'])['wght'].transform(
    lambda x: x * len(x) / x.sum()
)
    #  groupby(['country', 'year']) --> divide la columna wght en 15 grupos (5 países × 3 años)
    # transform devuelve un valor por cada fila (cada obs recibe su peso reescalado calculado con los valores de su grupo)
    # lambda x: --> crea una función donde x es el input
        # x → los pesos originales del grupo (por ejemplo, todos los wght de "Australia 2020")
        # len(x) → cuántas personas hay en ese grupo (el n)
        # x.sum() → suma de los pesos del grupo
        # x * len(x) / x.sum() → aplica la fórmula a cada persona del grupo


# Verificar
print('country' in df.columns)
print('year' in df.columns)

# La suma de pesos por grupo debe ser igual al n del grupo
print(df.groupby(['country', 'year']).agg(                   # Divide en 15 grupos; agg calcula varias estadísticas a la vez por grupo
    n=('wght_rescaled', 'count'),                            # Crea columna llamada n que cuenta cuántas filas hay en cada grupo    
    suma_pesos=('wght_rescaled', 'sum')                      # Crea columna suma_pesos que suma los pesos reescalados de cada grupo
).round(2))
    

True
True
                   n  suma_pesos
country   year                  
Australia 2020  4012      4012.0
          2021  3834      3834.0
          2022  3999      3999.0
Canada    2020  4023      4023.0
          2021  4259      4259.0
          2022  4151      4151.0
Mexico    2020  4003      4003.0
          2021  5607      5607.0
          2022  5834      5834.0
UK        2020  3969      3969.0
          2021  3873      3873.0
          2022  3917      3917.0
USA       2020  4382      4382.0
          2021  7075      7075.0
          2022  6987      6987.0


C:\Users\kathi\AppData\Local\Temp\ipykernel_6360\231847170.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['wght_rescaled'] = df.groupby(['country', 'year'])['wght'].transform(


In [35]:
# Guardar base procesada
print(f"Analytical sample: {len(df)}")

cols_keep = ['country', 'year', 'age', 'age_DV', 'sex', 'educ_DV', 
             'eth_DV', 'income_adeq', 'wght', 'wght_rescaled',
             'fruit', 'cereal', 'dairy', 'chicken', 'foodprok']

df[cols_keep].to_csv("../data/processed/IFPS_foodprok_clean.csv", index=False)
print("Guardado!")

Analytical sample: 69925
Guardado!
